In [36]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [37]:
df = pd.read_json("Amazon_Fashion.jsonl", lines=True)
print(df.head())

   rating                 title  \
0       5         Pretty locket   
1       5                     A   
2       2             Two Stars   
3       1       Won’t buy again   
4       5  I LOVE these glasses   

                                                text images        asin  \
0  I think this locket is really pretty. The insi...     []  B00LOPVX74   
1                                              Great     []  B07B4JXK8D   
2  One of the stones fell out within the first 2 ...     []  B007ZSEQ4Q   
3  Crappy socks. Money wasted. Bought to wear wit...     []  B07F2BTFS9   
4  I LOVE these glasses!  They fit perfectly over...     []  B00PKRFU4O   

  parent_asin                       user_id               timestamp  \
0  B00LOPVX74  AGBFYI2DDIKXC5Y4FARTYDTQBMFQ 2020-01-09 00:06:34.489   
1  B07B4JXK8D  AFQLNQNQYFWQZPJQZS6V3NZU4QBQ 2020-12-20 01:04:06.701   
2  B007ZSEQ4Q  AHITBJSS7KYUBVZPX7M2WJCOIVKQ 2015-05-23 01:33:48.000   
3  B07F2BTFS9  AFVNEEPDEIH5SPUN5BWC6NKL3WNQ 2018-12-31

In [38]:
df.drop(columns=["images", "asin", "parent_asin", "helpful", "user_id","timestamp"], inplace=True, errors="ignore")
df.head()

,rating,title,text,helpful_vote,verified_purchase
0,5,Pretty locket,I think this locket is really pretty. The insi...,3,True
1,5,A,Great,0,True
2,2,Two Stars,One of the stones fell out within the first 2 ...,3,True
3,1,Won’t buy again,Crappy socks. Money wasted. Bought to wear wit...,2,True
4,5,I LOVE these glasses,I LOVE these glasses! They fit perfectly over...,0,True


In [39]:
df.shape

(2500939, 5)

In [40]:
#removing missin rows
df = df.dropna(subset=['text'])

In [41]:
#duplicate rows dropped
df = df.drop_duplicates(['text','title'])

In [42]:
df.shape

(2349154, 5)

In [43]:
df['review'] = df['title'] + ' ' + df['text']

In [44]:
df.head()

,rating,title,text,helpful_vote,verified_purchase,review
0,5,Pretty locket,I think this locket is really pretty. The insi...,3,True,Pretty locket I think this locket is really pr...
1,5,A,Great,0,True,A Great
2,2,Two Stars,One of the stones fell out within the first 2 ...,3,True,Two Stars One of the stones fell out within th...
3,1,Won’t buy again,Crappy socks. Money wasted. Bought to wear wit...,2,True,Won’t buy again Crappy socks. Money wasted. Bo...
4,5,I LOVE these glasses,I LOVE these glasses! They fit perfectly over...,0,True,I LOVE these glasses I LOVE these glasses! Th...


In [45]:
df['review'][0]

'Pretty locket I think this locket is really pretty. The inside back is a solid silver depression and the front is a dome that is not solid (knotted). You could use it to store a small photo, lock of hair, etc but I use it when I need to carry medication with me. Closes securely. High quality & very pretty.'

In [46]:
def sentiment_label(rating):
    if rating >= 4:
        return 'positive'
    elif rating == 3:
        return 'neutral'
    else:
        return 'negative'
    
df['sentiment'] = df['rating'].apply(sentiment_label)

In [47]:
df.head()

,rating,title,text,helpful_vote,verified_purchase,review,sentiment
0,5,Pretty locket,I think this locket is really pretty. The insi...,3,True,Pretty locket I think this locket is really pr...,positive
1,5,A,Great,0,True,A Great,positive
2,2,Two Stars,One of the stones fell out within the first 2 ...,3,True,Two Stars One of the stones fell out within th...,negative
3,1,Won’t buy again,Crappy socks. Money wasted. Bought to wear wit...,2,True,Won’t buy again Crappy socks. Money wasted. Bo...,negative
4,5,I LOVE these glasses,I LOVE these glasses! They fit perfectly over...,0,True,I LOVE these glasses I LOVE these glasses! Th...,positive


In [48]:
print(df['sentiment'].value_counts())

sentiment
positive    1648267
negative     462820
neutral      238067
Name: count, dtype: int64


In [49]:
import re

def clean_review(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df['cleaned_review'] = df['review'].apply(clean_review)

In [50]:
df.head()

,rating,title,text,helpful_vote,verified_purchase,review,sentiment,cleaned_review
0,5,Pretty locket,I think this locket is really pretty. The insi...,3,True,Pretty locket I think this locket is really pr...,positive,pretty locket i think this locket is really pr...
1,5,A,Great,0,True,A Great,positive,a great
2,2,Two Stars,One of the stones fell out within the first 2 ...,3,True,Two Stars One of the stones fell out within th...,negative,two stars one of the stones fell out within th...
3,1,Won’t buy again,Crappy socks. Money wasted. Bought to wear wit...,2,True,Won’t buy again Crappy socks. Money wasted. Bo...,negative,won t buy again crappy socks money wasted boug...
4,5,I LOVE these glasses,I LOVE these glasses! They fit perfectly over...,0,True,I LOVE these glasses I LOVE these glasses! Th...,positive,i love these glasses i love these glasses they...


In [51]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()



[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [52]:
def preprocess(text):
    words = word_tokenize(text)
    words = [word for word in words if word not in stop_words]
    words = [lemmatizer.lemmatize(word) for word in words]

    return " ".join(words)

df['processed_review'] = df['cleaned_review'].apply(preprocess)

In [53]:
df.head()

,rating,title,text,helpful_vote,verified_purchase,review,sentiment,cleaned_review,processed_review
0,5,Pretty locket,I think this locket is really pretty. The insi...,3,True,Pretty locket I think this locket is really pr...,positive,pretty locket i think this locket is really pr...,pretty locket think locket really pretty insid...
1,5,A,Great,0,True,A Great,positive,a great,great
2,2,Two Stars,One of the stones fell out within the first 2 ...,3,True,Two Stars One of the stones fell out within th...,negative,two stars one of the stones fell out within th...,two star one stone fell within first week wear...
3,1,Won’t buy again,Crappy socks. Money wasted. Bought to wear wit...,2,True,Won’t buy again Crappy socks. Money wasted. Bo...,negative,won t buy again crappy socks money wasted boug...,buy crappy sock money wasted bought wear tieks...
4,5,I LOVE these glasses,I LOVE these glasses! They fit perfectly over...,0,True,I LOVE these glasses I LOVE these glasses! Th...,positive,i love these glasses i love these glasses they...,love glass love glass fit perfectly regular re...


In [67]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    df['processed_review'],
    df['sentiment'],
    test_size=0.2,
    random_state=42,
    stratify=df['sentiment']
)


In [68]:
X_train.shape

(1879323,)

In [69]:
print(type(X_train))

<class 'pandas.Series'>


In [98]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=5)

X_train_tfidf=tfidf.fit_transform(X_train)
X_test_tfidf=tfidf.transform(X_test)

In [99]:
X_train

757724     slow arrive nice shipping long like longer thi...
1318649    busted seam bought large well looking size cha...
394540                                  two star lightweight
1048483                    beautiful thanks beautiful thanks
1066619    big hit love dress inexpensive big hit buy pet...
                                 ...                        
690646     think looked better online ok teen think looke...
452264     tested notoriously stinky foot far aesthetic c...
90826      turn green light blue look quite cool imagined...
2067092    buy another one use look like picture material...
1148636    fantastic bag love bag want may give away gift...
Name: processed_review, Length: 1879323, dtype: str

In [100]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](3,)","[ 370256., 190454.,1318613.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](3,)","[-1.62,-2.29,-0.35]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[<U8](3,)","['negative','neutral','positive']"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](3, 5000)","[[ 38.59, 789.16, 84.19,..., 435.21, 24.3 , 24.57], [ 24.36, 488.39, 51.79,..., 95.76, 19.28, 27.06], [ 232.47,2212.74, 186.24,..., 45.92, 169.73, 317.05]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](3, 5000)","[[-10.39, -7.4 , -9.62,..., -7.99,-10.84,-10.83], [-10.23, -7.27, -9.5 ,..., -8.89,-10.46,-10.13], [ -9.89, -7.64,-10.11,...,-11.5 ,-10.2 , -9.58]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,5000


In [101]:
from sklearn.metrics import confusion_matrix,accuracy_score,classification_report
y_pred_tfidf=model.predict(X_test_tfidf)

In [102]:
confusion_matrix(y_test,y_pred_tfidf)

array([[ 67402,   2510,  22652],
       [ 11687,  10930,  24996],
       [  5636,   3744, 320274]])

In [103]:
print("TFIDF accuracy: ",accuracy_score(y_test,y_pred_tfidf))

TFIDF accuracy:  0.8484029363749944


In [104]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_tfidf))

              precision    recall  f1-score   support

    negative       0.80      0.73      0.76     92564
     neutral       0.64      0.23      0.34     47613
    positive       0.87      0.97      0.92    329654

    accuracy                           0.85    469831
   macro avg       0.77      0.64      0.67    469831
weighted avg       0.83      0.85      0.83    469831



In [105]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    max_iter=1000,
    random_state=42
)

lr.fit(X_train_tfidf, y_train)

y_pred = lr.predict(X_test_tfidf)

In [106]:
print(classification_report(y_test, y_pred))
cm = confusion_matrix(y_test, y_pred)
print(cm)

              precision    recall  f1-score   support

    negative       0.78      0.83      0.81     92564
     neutral       0.62      0.31      0.41     47613
    positive       0.91      0.96      0.94    329654

    accuracy                           0.87    469831
   macro avg       0.77      0.70      0.72    469831
weighted avg       0.86      0.87      0.86    469831

[[ 76884   4199  11481]
 [ 13932  14641  19040]
 [  7212   4790 317652]]


In [107]:
import joblib

joblib.dump(lr, "sentiment_model.pkl")
joblib.dump(tfidf, "tfidf.pkl")

['tfidf.pkl']